# Attack success rate versus input tokens

This notebook compares attack success rate (ASR) against the total number of input tokens in the attacked-ranking prompts. ASR counts successes over every requested passage, so invalid attacks remain in the denominator. Circles are ordinary attacks; squares are defended evaluations. The notebook displays figures inline and does not write image files.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

RESULTS = Path.cwd()
if not (RESULTS / 'attack_outcomes.csv').exists():
    RESULTS = Path.cwd() / 'Results'

OUTCOMES_PATH = RESULTS / 'attack_outcomes.csv'
TOKENS_PATH = RESULTS / 'prompt_token_counts.csv'

if not TOKENS_PATH.exists():
    raise FileNotFoundError(
        f'{TOKENS_PATH} is missing. Run Results/calculate_prompt_tokens.py first.'
    )

outcomes = pd.read_csv(OUTCOMES_PATH)
tokens = pd.read_csv(TOKENS_PATH)
outcomes.shape, tokens.shape

## Join latest ASR outcomes to attacked prompt-token totals

The token script selects the newest usable raw result for each run configuration. This cell keeps only its attacked-prompt rows and joins them to the ASR table by dataset, model, ranking paradigm, attack, and prompt mode.

In [ ]:
JOIN_COLUMNS = ['Dataset', 'Model', 'Paradigm', 'Attack', 'Prompt']

attacked_tokens = tokens.loc[
    (tokens['Phase'] == 'Attacked') & (tokens['Prompts counted'] > 0),
    JOIN_COLUMNS + ['Total tokens', 'Prompts counted', 'Prompts missing text', 'Source'],
].copy()
attacked_tokens = attacked_tokens.rename(
    columns={
        'Total tokens': 'Input tokens',
        'Prompts counted': 'Prompts tokenized',
        'Prompts missing text': 'Prompts without saved text',
        'Source': 'Token source',
    }
)

plot_data = outcomes.merge(attacked_tokens, on=JOIN_COLUMNS, how='inner')
plot_data['ASR (%)'] = 100 * plot_data['Attack success'] / plot_data['Requested']
plot_data['Evaluation'] = plot_data['Prompt'].eq('Default').map(
    {True: 'Attack', False: 'Defense'}
)
plot_data['Marker'] = plot_data['Evaluation'].map({'Attack': 'o', 'Defense': 's'})

plot_data = plot_data.sort_values(JOIN_COLUMNS).drop_duplicates(JOIN_COLUMNS, keep='last')
plot_data[JOIN_COLUMNS + ['Input tokens', 'ASR (%)', 'Evaluation']].sort_values(JOIN_COLUMNS)

## ASR versus attacked input tokens

Each panel is an attack type and ranking paradigm. Colour identifies the model, while shape identifies whether the ranking used the ordinary attack prompt or a defense prompt.

In [ ]:
sns.set_theme(style='whitegrid', context='talk')

if plot_data.empty:
    raise ValueError('No ASR rows could be matched to attacked prompt-token totals.')

panel_data = plot_data.copy()
panel_data['Panel'] = panel_data['Attack'] + ' — ' + panel_data['Paradigm']
panels = list(panel_data['Panel'].drop_duplicates())
ncols = min(3, len(panels))
nrows = (len(panels) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows), squeeze=False)
palette = dict(zip(sorted(panel_data['Model'].unique()), sns.color_palette('tab10')))

for axis, panel in zip(axes.flat, panels):
    subset = panel_data.loc[panel_data['Panel'] == panel]
    for evaluation, marker in [('Attack', 'o'), ('Defense', 's')]:
        points = subset.loc[subset['Evaluation'] == evaluation]
        axis.scatter(
            points['Input tokens'],
            points['ASR (%)'],
            c=points['Model'].map(palette),
            marker=marker,
            s=95,
            edgecolors='black',
            linewidths=0.6,
            alpha=0.9,
            label=evaluation,
        )
    axis.set_title(panel)
    axis.set_xlabel('Input tokens across attacked prompts')
    axis.set_ylabel('ASR (%)')
    axis.set_ylim(-2, 102)

for axis in axes.flat[len(panels):]:
    axis.set_visible(False)

model_handles = [
    plt.Line2D([], [], marker='o', linestyle='', color=color, label=model, markersize=8)
    for model, color in palette.items()
]
shape_handles = [
    plt.Line2D([], [], marker='o', linestyle='', color='black', label='Attack', markersize=8),
    plt.Line2D([], [], marker='s', linestyle='', color='black', label='Defense', markersize=8),
]
fig.legend(handles=model_handles + shape_handles, loc='lower center', ncol=min(5, len(model_handles) + 2))
fig.suptitle('Attack success rate versus locally estimated input tokens', y=1.02)
fig.tight_layout(rect=(0, 0.08, 1, 1))
plt.show()

## Coverage check

Rows below lack saved attacked prompt text or did not have a matching ASR outcome. They are excluded from the graph rather than treated as zero tokens.

In [ ]:
unmatched_outcomes = outcomes.merge(
    attacked_tokens[JOIN_COLUMNS], on=JOIN_COLUMNS, how='left', indicator=True
).query("_merge == 'left_only'")

coverage = {
    'ASR rows': len(outcomes),
    'Rows plotted': len(plot_data),
    'ASR rows without attacked-token totals': len(unmatched_outcomes),
    'Attacked prompts without saved text': int(attacked_tokens['Prompts without saved text'].sum()),
}
pd.Series(coverage, name='Count')